# Test Assistant-To-Strategy Token Extraction

This notebook uses helpers from `extract_activations.py` to find the span from the generated `assistant` marker through the generated `Strategy:` text before `Steps:`, map it to token positions, and decode those tokens back to text.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
while not (project_root / "src").is_dir():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from scripts.experiments.feature_geometry_new import extract_activations as ea

records = ea.load_completion_records(ea.DEFAULT_INPUT_PATH, max_samples=20)
record = next(r for r in records if ea.find_assistant_strategy_span(r["full_text"]) is not None)
span = ea.find_assistant_strategy_span(record["full_text"])
span

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(ea.DEFAULT_MODEL_NAME)

start_char, end_char, activation_text = span
token_positions = ea.token_positions_for_char_span(
    tokenizer,
    record["full_text"],
    start_char,
    end_char,
)

len(token_positions), token_positions[:10], token_positions[-10:]

In [ ]:
encoded = tokenizer(
    record["full_text"],
    return_offsets_mapping=True,
    add_special_tokens=True,
)
input_ids = encoded["input_ids"]
selected_token_ids = [input_ids[pos] for pos in token_positions]
decoded_text = tokenizer.decode(selected_token_ids, skip_special_tokens=True)

print("Original activation text:")
print(activation_text)
print("\nDecoded selected tokens:")
print(decoded_text)

In [ ]:
print("Prompt-side Steps index:", record["full_text"].find("Steps:"))
print("Generated span starts at:", start_char)
print("Generated span starts with assistant:", activation_text.startswith("assistant"))
print("Generated span contains Steps:", "Steps:" in activation_text)
print("Decoded contains Steps:", "Steps:" in decoded_text)
print("Decoded contains numbered step content:", "\n1." in decoded_text)

assert decoded_text.startswith("assistant")
assert "Steps:" not in decoded_text
assert "\n1." not in decoded_text